In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader

# ── Data augmentation — makes model more robust ───────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])


train_data = datasets.CIFAR10('./data', train=True,
                               download=True,
                               transform=transform_train)
test_data  = datasets.CIFAR10('./data', train=False,
                               transform=transform_test)

train_loader = DataLoader(train_data, batch_size=64,
                           shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_data,  batch_size=256,
                           num_workers=2)


# ── Strategy 1: Feature extractor ────────────────────────
def build_feature_extractor():
    # Use weights= instead of pretrained= (fixes the warning)
    model = models.resnet18(
        weights=models.ResNet18_Weights.DEFAULT
    )
    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace head — CIFAR has 10 classes
    model.fc = nn.Linear(512, 10)

    trainable = sum(p.numel() for p in model.parameters()
                    if p.requires_grad)
    print(f"Feature extractor: {trainable:,} params train")
    return model


# ── Strategy 2: Fine-tuned ────────────────────────────────
def build_fine_tuned():
    model = models.resnet18(
        weights=models.ResNet18_Weights.DEFAULT
    )
    # Only freeze early layers
    for name, param in model.named_parameters():
        if 'layer4' not in name and 'fc' not in name:
            param.requires_grad = False

    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 10)
    )
    trainable = sum(p.numel() for p in model.parameters()
                    if p.requires_grad)
    print(f"Fine-tuned: {trainable:,} params train")
    return model


# ── Strategy 3: Full fine-tune ────────────────────────────
def build_full_finetune():
    model = models.resnet18(
        weights=models.ResNet18_Weights.DEFAULT
    )
    model.fc = nn.Linear(512, 10)
    print("Full fine-tune: all params train")
    return model


# ── Training function ─────────────────────────────────────
def train_model(model, name, epochs=5):
    # Different LR for pretrained vs new layers
    if name == 'full':
        optimizer = torch.optim.Adam([
            {'params': [p for n, p in model.named_parameters()
                        if 'fc' not in n and p.requires_grad],
             'lr': 1e-4},
            {'params': model.fc.parameters(), 'lr': 1e-3}
        ])
    else:
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad,
                   model.parameters()),
            lr=1e-3
        )

    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )

    best_acc = 0
    print(f"\nTraining {name}...\n")

    for epoch in range(epochs):
        # Train
        model.train()
        total_loss, correct, total = 0, 0, 0

        for X, y in train_loader:
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (out.argmax(1) == y).sum().item()
            total      += len(y)

        # Evaluate
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                out = model(X)
                val_correct += (out.argmax(1)==y).sum().item()
                val_total   += len(y)

        test_acc = val_correct / val_total
        scheduler.step()

        if test_acc > best_acc:
            best_acc = test_acc

        print(f"Epoch {epoch+1} | "
              f"Loss: {total_loss/len(train_loader):.4f} | "
              f"Train: {correct/total:.4f} | "
              f"Test: {test_acc:.4f}")

    print(f"Best: {best_acc:.4f}\n")
    return best_acc


# ── Run all 3 — only 5 epochs each ───────────────────────
results = {}

results['feature_extractor'] = train_model(
    build_feature_extractor(), 'feature_extractor', epochs=5
)
results['fine_tuned'] = train_model(
    build_fine_tuned(), 'fine_tuned', epochs=5
)
results['full_finetune'] = train_model(
    build_full_finetune(), 'full', epochs=5
)

print("=== Strategy Comparison ===")
for name, acc in results.items():
    bar = '█' * int(acc * 50)
    print(f"  {name:20s}: {acc:.4f}  {bar}")

Feature extractor: 5,130 params train

Training feature_extractor...

Epoch 1 | Loss: 1.8240 | Train: 0.3574 | Test: 0.3925
Epoch 2 | Loss: 1.7206 | Train: 0.3952 | Test: 0.3975
Epoch 3 | Loss: 1.6953 | Train: 0.4018 | Test: 0.4088
Epoch 4 | Loss: 1.6789 | Train: 0.4119 | Test: 0.4146
Epoch 5 | Loss: 1.6558 | Train: 0.4201 | Test: 0.4173
Best: 0.4173

Fine-tuned: 8,527,626 params train

Training fine_tuned...

Epoch 1 | Loss: 1.3014 | Train: 0.5481 | Test: 0.6322
Epoch 2 | Loss: 1.1039 | Train: 0.6207 | Test: 0.6583
Epoch 3 | Loss: 1.0193 | Train: 0.6437 | Test: 0.6695
Epoch 4 | Loss: 0.9435 | Train: 0.6721 | Test: 0.6891
Epoch 5 | Loss: 0.8839 | Train: 0.6905 | Test: 0.6976
Best: 0.6976

Full fine-tune: all params train

Training full...

Epoch 1 | Loss: 1.1569 | Train: 0.5955 | Test: 0.7274
Epoch 2 | Loss: 0.7825 | Train: 0.7290 | Test: 0.7687
Epoch 3 | Loss: 0.6600 | Train: 0.7730 | Test: 0.7979
Epoch 4 | Loss: 0.5842 | Train: 0.7965 | Test: 0.8148
Epoch 5 | Loss: 0.5284 | Train: 0.